# 08 — Inference Overhead

Cost of the two-stage approach relative to flat classification, measured on the main models from notebooks 01–02 without further training: model size, inference latency per image on GPU and CPU, the proportion of test images routed to stage 2, and training time.

## 1. Configuration

In [8]:
import os, time, platform
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from pathlib import Path
from PIL import Image
from torchvision import models, transforms
from google.colab import drive
drive.mount('/content/drive')

# Paths (adjust to your Google Drive)
DATA_DIR = Path('/content/drive/MyDrive/USK-Coffee')
OUT_ROOT = Path('/content/drive/MyDrive/coffee-two-stage/models')  # models from notebook 01
OH_OUT   = Path('/content/drive/MyDrive/coffee-two-stage/inference_overhead')
OH_OUT.mkdir(parents=True, exist_ok=True)

ARCHS = ['resnet50', 'efficientnet_b0']
SEEDS = [0, 1, 2]
SCHEMES = {'4class': 4, '3class': 3, '2class': 2}   # flat, stage 1, stage 2
IMG_SIZE = 224
MEAN, STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]
EXTS = {'.jpg', '.jpeg', '.png', '.bmp'}
TIMING = {'cuda': (50, 60), 'cpu': (10, 20)}   # (warm-up runs, timed runs per round) per model
ROUNDS = 5
torch.set_num_threads(1)   # fixed CPU thread count, reported with the results

def run_dir(arch, scheme, seed):
    return OUT_ROOT / arch / scheme / f'seed{seed}'

assert torch.cuda.is_available(), 'select a GPU runtime'
cpu_name = next((l.split(':', 1)[1].strip() for l in open('/proc/cpuinfo') if l.startswith('model name')),
                platform.processor())
print('GPU:', torch.cuda.get_device_name(0))
print('CPU:', cpu_name, '| threads:', torch.get_num_threads())
print('torch', torch.__version__)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
GPU: Tesla T4
CPU: Intel(R) Xeon(R) CPU @ 2.00GHz | threads: 1
torch 2.14.1+cu130


## 2. Models

In [9]:
def build_model(arch, num_classes):
    if arch == 'resnet50':
        m = models.resnet50(weights=None)
        m.fc = nn.Linear(m.fc.in_features, num_classes)
    elif arch == 'efficientnet_b0':
        m = models.efficientnet_b0(weights=None)
        m.classifier[1] = nn.Linear(m.classifier[1].in_features, num_classes)
    else:
        raise ValueError(arch)
    return m

def load_model(arch, scheme, seed=0):
    m = build_model(arch, SCHEMES[scheme])
    m.load_state_dict(torch.load(run_dir(arch, scheme, seed) / 'final_model.pt', map_location='cpu'))
    return m.eval()

## 3. Model size

In [10]:
rows = []
for arch in ARCHS:
    for scheme in SCHEMES:
        m = load_model(arch, scheme)
        rows.append({'arch': arch, 'scheme': scheme,
                     'params_M': sum(p.numel() for p in m.parameters()) / 1e6,
                     'file_MB': os.path.getsize(run_dir(arch, scheme, 0) / 'final_model.pt') / 2**20})
size = pd.DataFrame(rows)
size.to_csv(OH_OUT / 'model_size.csv', index=False)
print(size.round(3).to_string(index=False))

           arch scheme  params_M  file_MB
       resnet50 4class    23.516   90.011
       resnet50 3class    23.514   90.004
       resnet50 2class    23.512   89.996
efficientnet_b0 4class     4.013   15.593
efficientnet_b0 3class     4.011   15.588
efficientnet_b0 2class     4.010   15.583


## 4. Inference latency

In [11]:
eval_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])
img_path = sorted(f for f in (DATA_DIR / 'test' / 'premium').iterdir() if f.suffix.lower() in EXTS)[0]
with Image.open(img_path) as img:
    x_cpu = eval_tf(img.convert('RGB')).unsqueeze(0)   # one preprocessed image, batch size 1

# Forward pass only (image loading and preprocessing excluded)
@torch.inference_mode()
def time_runs(model, x, device, n_runs):
    times = []
    for _ in range(n_runs):
        if device == 'cuda':
            torch.cuda.synchronize()
        t0 = time.perf_counter()
        model(x)
        if device == 'cuda':
            torch.cuda.synchronize()   # wait for the GPU before stopping the clock
        times.append((time.perf_counter() - t0) * 1000)
    return times

# Latency does not depend on the trained weights, so the seed-0 models are used.
# The three models of an architecture are timed in rounds, in a random order per round,
# so that no model is always measured first.
rng = np.random.default_rng(0)
rows = []
for device, (n_warm, n_runs) in TIMING.items():
    x = x_cpu.to(device)
    for arch in ARCHS:
        loaded = {s: load_model(arch, s).to(device) for s in SCHEMES}
        for m in loaded.values():
            time_runs(m, x, device, n_warm)
        times = {s: [] for s in SCHEMES}
        for _ in range(ROUNDS):
            for s in rng.permutation(list(SCHEMES)):
                times[str(s)] += time_runs(loaded[str(s)], x, device, n_runs)
        for s, t in times.items():
            med, q1, q3 = np.median(t), np.percentile(t, 25), np.percentile(t, 75)
            rows.append({'device': device, 'arch': arch, 'scheme': s,
                         'median_ms': med, 'q1_ms': q1, 'q3_ms': q3, 'n_runs': len(t)})
            print(f'{device:>4} | {arch:<15} | {s} | {med:.2f} ms (IQR {q1:.2f}-{q3:.2f})')
        del loaded
        if device == 'cuda':
            torch.cuda.empty_cache()
lat = pd.DataFrame(rows)
lat.to_csv(OH_OUT / 'latency.csv', index=False)

cuda | resnet50        | 4class | 7.12 ms (IQR 5.86-7.56)
cuda | resnet50        | 3class | 6.94 ms (IQR 5.86-7.86)
cuda | resnet50        | 2class | 6.89 ms (IQR 5.79-7.59)
cuda | efficientnet_b0 | 4class | 8.06 ms (IQR 7.87-8.36)
cuda | efficientnet_b0 | 3class | 8.12 ms (IQR 7.92-8.85)
cuda | efficientnet_b0 | 2class | 7.97 ms (IQR 7.81-8.22)
 cpu | resnet50        | 4class | 120.55 ms (IQR 116.04-159.86)
 cpu | resnet50        | 3class | 117.96 ms (IQR 115.62-130.36)
 cpu | resnet50        | 2class | 130.57 ms (IQR 118.50-162.01)
 cpu | efficientnet_b0 | 4class | 36.58 ms (IQR 33.56-45.38)
 cpu | efficientnet_b0 | 3class | 35.36 ms (IQR 33.68-44.87)
 cpu | efficientnet_b0 | 2class | 34.14 ms (IQR 33.60-41.32)


## 5. Proportion of test images routed to stage 2

In [12]:
rows = []
for arch in ARCHS:
    for seed in SEEDS:
        z = np.load(run_dir(arch, '3class', seed) / 'test_predictions.npz')
        rows.append({'arch': arch, 'seed': seed,
                     'routed_share': float((z['prob'].argmax(1) == 2).mean())})   # 2 = merged peaberry-premium
routed = pd.DataFrame(rows)
routed.to_csv(OH_OUT / 'routed_share.csv', index=False)
print(routed.round(4).to_string(index=False))
print(routed.groupby('arch')['routed_share'].agg(['mean', 'std']).round(4))

           arch  seed  routed_share
       resnet50     0        0.5706
       resnet50     1        0.5581
       resnet50     2        0.5894
efficientnet_b0     0        0.5638
efficientnet_b0     1        0.5406
efficientnet_b0     2        0.5412
                   mean     std
arch                           
efficientnet_b0  0.5485  0.0132
resnet50         0.5727  0.0157


## 6. Training time of the main models

In [13]:
# Sum of per-epoch time in history.csv (training and validation of each epoch)
rows = []
for arch in ARCHS:
    for scheme in SCHEMES:
        for seed in SEEDS:
            h = pd.read_csv(run_dir(arch, scheme, seed) / 'history.csv')
            assert 'sec' in h.columns, f'no epoch time in {arch}/{scheme}/seed{seed}'
            rows.append({'arch': arch, 'scheme': scheme, 'seed': seed,
                         'epochs_run': len(h), 'train_minutes': h['sec'].sum() / 60})
train = pd.DataFrame(rows)
train.to_csv(OH_OUT / 'training_time_main.csv', index=False)

per_seed = train.pivot_table(index=['arch', 'seed'], columns='scheme', values='train_minutes')
per_seed['two_stage'] = per_seed['3class'] + per_seed['2class']
print(per_seed.round(1).to_string())
print(per_seed.groupby('arch')[['4class', 'two_stage']].agg(['mean', 'std']).round(1))

scheme                2class  3class  4class  two_stage
arch            seed                                   
efficientnet_b0 0        6.8    11.2    13.7       18.0
                1       10.1    19.6    21.2       29.7
                2        4.3    10.1    11.0       14.4
resnet50        0       14.0    17.3    14.7       31.3
                1        6.5    11.1    18.3       17.6
                2       10.0    13.3    15.2       23.2
scheme          4class      two_stage     
                  mean  std      mean  std
arch                                      
efficientnet_b0   15.3  5.3      20.7  8.0
resnet50          16.1  1.9      24.0  6.9


## 7. Summary

In [14]:
sz = size.set_index(['arch', 'scheme'])
lt = lat.set_index(['device', 'arch', 'scheme'])['median_ms']
share = routed.groupby('arch')['routed_share'].mean()
tt = per_seed.groupby('arch')[['4class', 'two_stage']].mean()

rows = []
for arch in ARCHS:
    for device in TIMING:
        t_flat, t1, t2 = (lt[(device, arch, s)] for s in SCHEMES)
        t_two = t1 + share[arch] * t2   # stage 2 runs only for images routed to it
        rows.append({'arch': arch, 'device': device,
                     'flat_ms': t_flat, 'two_stage_ms': t_two, 'two_stage_worst_ms': t1 + t2,
                     'latency_ratio': t_two / t_flat, 'routed_share': share[arch],
                     'flat_params_M': sz.loc[(arch, '4class'), 'params_M'],
                     'two_stage_params_M': sz.loc[(arch, '3class'), 'params_M'] + sz.loc[(arch, '2class'), 'params_M'],
                     'flat_MB': sz.loc[(arch, '4class'), 'file_MB'],
                     'two_stage_MB': sz.loc[(arch, '3class'), 'file_MB'] + sz.loc[(arch, '2class'), 'file_MB'],
                     'flat_train_min': tt.loc[arch, '4class'], 'two_stage_train_min': tt.loc[arch, 'two_stage']})
summary = pd.DataFrame(rows)
summary.to_csv(OH_OUT / 'overhead_summary.csv', index=False)
pd.set_option('display.width', 220)
print(summary.round(2).to_string(index=False))

           arch device  flat_ms  two_stage_ms  two_stage_worst_ms  latency_ratio  routed_share  flat_params_M  two_stage_params_M  flat_MB  two_stage_MB  flat_train_min  two_stage_train_min
       resnet50   cuda     7.12         10.89               13.84           1.53          0.57          23.52               47.03    90.01        180.00           16.07                24.04
       resnet50    cpu   120.55        192.74              248.53           1.60          0.57          23.52               47.03    90.01        180.00           16.07                24.04
efficientnet_b0   cuda     8.06         12.49               16.09           1.55          0.55           4.01                8.02    15.59         31.17           15.30                20.67
efficientnet_b0    cpu    36.58         54.09               69.50           1.48          0.55           4.01                8.02    15.59         31.17           15.30                20.67
